In [3]:
import pandas as pd

raw_path = "../Raw_Data/incident_event_log.csv"

df = pd.read_csv(raw_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (141712, 36)


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


In [4]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

Total rows: 141712
Total columns: 36

Column names:
['number', 'incident_state', 'active', 'reassignment_count', 'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by', 'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by', 'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory', 'u_symptom', 'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group', 'assigned_to', 'knowledge', 'u_priority_confirmation', 'notify', 'problem_id', 'rfc', 'vendor', 'caused_by', 'closed_code', 'resolved_by', 'resolved_at', 'closed_at']


In [5]:
import pandas as pd
import numpy as np

In [6]:
raw_path = "../Raw_Data/incident_event_log.csv"

df = pd.read_csv(raw_path)

print("Dataset shape:", df.shape)

Dataset shape: (141712, 36)


In [7]:
df_work = df.copy()

print("Working copy shape:", df_work.shape)

Working copy shape: (141712, 36)


In [8]:

# Columns we do not want as model features
columns_to_remove = [
    "number",
    "incident_state",
    "active",
    "reassignment_count",
    "reopen_count",
    "sys_mod_count",
    "made_sla",
    "caller_id",
    "opened_by",
    "opened_at",
    "sys_created_by",
    "sys_created_at",
    "sys_updated_by",
    "sys_updated_at",
    "cmdb_ci",
    "priority",
    "assignment_group",
    "assigned_to",
    "knowledge",
    "u_priority_confirmation",
    "notify",
    "problem_id",
    "rfc",
    "vendor",
    "caused_by",
    "closed_code",
    "resolved_by",
    "resolved_at",
    "closed_at"
]

# Remove only columns that exist in the dataset
df_work = df_work.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Shape after feature removal:", df_work.shape)
print("Remaining columns:", df_work.columns.tolist())


Shape after feature removal: (141712, 7)
Remaining columns: ['contact_type', 'location', 'category', 'subcategory', 'u_symptom', 'impact', 'urgency']


In [9]:
# STEP 2: CHECK FOR DUPLICATE ROWS

# Count rows that are completely identical across all remaining columns
duplicate_count = df_work.duplicated().sum()

# Display the total number of rows and duplicate rows
print("Total rows:", len(df_work))
print("Fully duplicated rows:", duplicate_count)

Total rows: 141712
Fully duplicated rows: 132601


In [10]:
# STEP 3: CHECK DUPLICATES IN THE ORIGINAL DATASET

# Count rows that are identical across all original columns
original_duplicate_count = df.duplicated().sum()

print("Original dataset rows:", len(df))
print("Exact duplicates in original data:", original_duplicate_count)


Original dataset rows: 141712
Exact duplicates in original data: 0


In [11]:
# STEP 3: KEEP ONE RECORD PER INCIDENT

# Sort events so the earliest recorded event comes first
df_work = df.sort_values(
    by="sys_created_at",
    kind="stable"
).copy()

# Keep only the first event for each unique incident ID
df_work = df_work.drop_duplicates(
    subset="number",
    keep="first"
)

# Display the new dataset size
print("Rows after removing repeated incident events:", len(df_work))
print("Unique incidents remaining:", df_work["number"].nunique())

Rows after removing repeated incident events: 24918
Unique incidents remaining: 24918


In [12]:
# STEP 4: CHECK MISSING VALUES
# Count missing values in each column of the working dataset

missing_values = df_work.isnull().sum()

# Display only columns that contain missing values
print(missing_values[missing_values > 0].sort_values(ascending=False))

Series([], dtype: int64)


In [13]:
# STEP 5: CHECK FOR QUESTION MARKS
# In this dataset, '?' may represent missing information.
# Count how many '?' values remain in each column.

question_marks = (df_work == "?").sum()

# Display only columns that contain '?'
print(question_marks[question_marks > 0].sort_values(ascending=False))

caused_by           24916
vendor              24903
cmdb_ci             24867
rfc                 24820
problem_id          24677
sys_created_by      11495
sys_created_at      11495
assigned_to          7140
u_symptom            5735
resolved_at          1556
assignment_group      906
opened_by             714
closed_code           107
resolved_by            99
subcategory            69
location               52
category               39
caller_id               3
dtype: int64


In [15]:
# STEP 6: CONVERT '?' INTO MISSING VALUES
# The dataset uses '?' to indicate unavailable information.
# Replace these markers with Pandas' standard missing value (NaN).

df_work = df_work.replace("?", np.nan)

# Count missing values in each column
missing_values = df_work.isnull().sum()

# Display columns that contain missing values
print(missing_values[missing_values > 0].sort_values(ascending=False))

caused_by           24916
vendor              24903
cmdb_ci             24867
rfc                 24820
problem_id          24677
sys_created_by      11495
sys_created_at      11495
assigned_to          7140
u_symptom            5735
resolved_at          1556
assignment_group      906
opened_by             714
closed_code           107
resolved_by            99
subcategory            69
location               52
category               39
caller_id               3
dtype: int64


In [16]:

# STEP 7: REMOVE UNNECESSARY COLUMNS
# Remove identifiers, IT-specific fields, and information
# that would not be available when an incident is first reported.

columns_to_remove = [
    "number",
    "caller_id",
    "opened_by",
    "sys_created_by",
    "sys_created_at",
    "sys_updated_by",
    "sys_updated_at",
    "incident_state",
    "active",
    "reassignment_count",
    "reopen_count",
    "sys_mod_count",
    "made_sla",
    "cmdb_ci",
    "priority",
    "assignment_group",
    "assigned_to",
    "knowledge",
    "u_priority_confirmation",
    "notify",
    "problem_id",
    "rfc",
    "vendor",
    "caused_by",
    "closed_code",
    "resolved_by",
    "resolved_at",
    "closed_at"
]

# Remove only columns that exist in the working dataset
df_work = df_work.drop(
    columns=columns_to_remove,
    errors="ignore"
)

# Display the remaining columns
print("Shape after removing columns:", df_work.shape)
print("Remaining columns:")
print(df_work.columns.tolist())


Shape after removing columns: (24918, 8)
Remaining columns:
['opened_at', 'contact_type', 'location', 'category', 'subcategory', 'u_symptom', 'impact', 'urgency']


In [17]:
# STEP 8: INSPECT REMAINING COLUMNS
# Check data types and non-missing values in each column

df_work.info()

<class 'pandas.DataFrame'>
Index: 24918 entries, 140890 to 141708
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   opened_at     24918 non-null  str  
 1   contact_type  24918 non-null  str  
 2   location      24866 non-null  str  
 3   category      24879 non-null  str  
 4   subcategory   24849 non-null  str  
 5   u_symptom     19183 non-null  str  
 6   impact        24918 non-null  str  
 7   urgency       24918 non-null  str  
dtypes: str(8)
memory usage: 1.7 MB


In [18]:

# STEP 9A: FILL MISSING LOCATION VALUES
# Replace missing locations with "Unknown" instead of guessing a location.

df_work["location"] = df_work["location"].fillna("Unknown")

# Check how many missing values remain in the location column.
print("Missing location values:", df_work["location"].isnull().sum())


Missing location values: 0


In [19]:

# STEP 9B: FILL MISSING CATEGORY VALUES
# Replace missing categories with "Unknown" rather than guessing a category.

df_work["category"] = df_work["category"].fillna("Unknown")

# Check how many missing values remain in the category column.
print("Missing category values:", df_work["category"].isnull().sum())


Missing category values: 0


In [20]:

# STEP 9C: FILL MISSING SUBCATEGORY VALUES
# Replace missing subcategories with "Unknown" instead of guessing their values.

df_work["subcategory"] = df_work["subcategory"].fillna("Unknown")

# Check how many missing values remain in the subcategory column.
print("Missing subcategory values:", df_work["subcategory"].isnull().sum())


Missing subcategory values: 0


In [21]:

# STEP 9D: FILL MISSING SYMPTOM VALUES
# Replace missing symptom values with "Unknown".
# This preserves the records without inventing symptom information.

df_work["u_symptom"] = df_work["u_symptom"].fillna("Unknown")

# Check how many missing values remain in the symptom column.
print("Missing symptom values:", df_work["u_symptom"].isnull().sum())


Missing symptom values: 0


In [22]:

# STEP 10: VERIFY MISSING VALUES
# Count missing values in every remaining column.
# Ideally, every column should now have zero missing values.

print(df_work.isnull().sum())


opened_at       0
contact_type    0
location        0
category        0
subcategory     0
u_symptom       0
impact          0
urgency         0
dtype: int64


In [23]:

# STEP 11: CHECK UNIQUE VALUES
# Count how many distinct values exist in each remaining column.
# This helps us understand the categories available for model training.

for column in df_work.columns:
    print(column, ":", df_work[column].nunique(), "unique values")


opened_at : 19849 unique values
contact_type : 5 unique values
location : 220 unique values
category : 58 unique values
subcategory : 247 unique values
u_symptom : 501 unique values
impact : 3 unique values
urgency : 3 unique values


In [24]:

# STEP 12: CONVERT OPENED_AT TO DATETIME
# Convert the opened_at column from text into a proper date-time format.
# If any value cannot be converted, pandas will mark it as NaT (missing datetime).

df_work["opened_at"] = pd.to_datetime(
    df_work["opened_at"],
    errors="coerce",
    dayfirst=True
)

# Display the data type and a few converted values.
print("Data type:", df_work["opened_at"].dtype)
print(df_work["opened_at"].head())


Data type: datetime64[us]
140890   2017-01-01 02:15:00
140905   2017-01-01 21:32:00
140557   2016-11-01 10:52:00
140561   2016-11-01 12:21:00
140567   2016-11-01 14:23:00
Name: opened_at, dtype: datetime64[us]


In [25]:

# STEP 13: CHECK INVALID OR MISSING DATES
# Count dates that could not be converted into datetime format.

invalid_dates = df_work["opened_at"].isna().sum()

print("Invalid or missing dates:", invalid_dates)


Invalid or missing dates: 0
